In [1]:
!pip install git+https://github.com/huggingface/transformers

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-m7mo5fna
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-m7mo5fna
  Resolved https://github.com/huggingface/transformers to commit ffbea2db927e4a0fb4bc8615c909ae277d8df0e6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip install triton mistral-common --upgrade

In [1]:
import sys

sys.argv = [
    "program"
]


In [ ]:
from huggingface_hub import login

HF_TOKEN = 'hf_'
login(token=HF_TOKEN)

In [3]:
# scripts/02_emotion_direction_extraction/1_dump_residual_aligned_sublayer_activations.py
# -*- coding: utf-8 -*-
"""
Dump Residual-Aligned Sublayer Activations - Adapted for Ministral-3-3B

Performs forward passes using accepted samples from folder 01.
Saves attention and MLP layer outputs after they have been added back to the residual stream.
Used for calculating emotion vectors, saving data for all 6 emotions grouped together.
"""

import os, json, time, traceback, argparse
from pathlib import Path
import torch
import numpy as np
from transformers import AutoTokenizer, Mistral3ForConditionalGeneration
from huggingface_hub import login
from collections import defaultdict

# HF token (priority to env var, else default login)
HF_TOKEN = os.environ.get('HF_TOKEN', None)
if HF_TOKEN:
    login(token=HF_TOKEN)

# Emotion vector calculation constants
EMOS6 = ["anger","sadness","happiness","fear","surprise","disgust"]

# ============== Hook class to capture attention layer output ==============
class AttentionHook:
    """
    Capture attention layer output (before residual) for each layer - Adapted for Ministral-3-3B
    """
    def __init__(self, model):
        self.model = model
        self.hooks = []
        self.attention_outputs = {}  # layer_idx -> attention_output (before residual)
        self.register_hooks()

    def register_hooks(self):
        """
        Register hooks to attention layers - Ministral-3-3B specific
        """
        # Access the language model layers
        language_model = self.model.model.language_model

        # Find the layers container - based on Ministral3Model structure
        if hasattr(language_model, 'layers'):
            layers = language_model.layers
            print(f"Found {len(layers)} layers in language_model.layers")
        elif hasattr(language_model, 'decoder') and hasattr(language_model.decoder, 'layers'):
            layers = language_model.decoder.layers
            print(f"Found {len(layers)} layers in language_model.decoder.layers")
        else:
            # Fallback search for layers
            layers = None
            for name, module in language_model.named_children():
                if 'layer' in name.lower() and hasattr(module, '__len__'):
                    layers = module
                    print(f"Found layers at language_model.{name}")
                    break

        if layers is None:
            raise ValueError("Could not find transformer layers in the model")

        for layer_idx, layer in enumerate(layers):
            # Find the attention module
            attn_module = None
            if hasattr(layer, 'self_attn'):
                attn_module = layer.self_attn
            elif hasattr(layer, 'attention'):
                attn_module = layer.attention
            else:
                for name, module in layer.named_children():
                    if 'attn' in name.lower():
                        attn_module = module
                        break

            if attn_module is None:
                print(f"Warning: No attention module found in layer {layer_idx}")
                continue

            # Hook attention layer at the output
            def make_attention_hook(idx):
                def attention_hook(module, input, output):
                    # For attention layer, output[0] is attention output (before residual)
                    if isinstance(output, tuple) and len(output) > 0:
                        attention_output = output[0]
                        self.attention_outputs[idx] = attention_output.detach().cpu()
                    elif isinstance(output, torch.Tensor):
                        self.attention_outputs[idx] = output.detach().cpu()
                return attention_hook

            # Register hook
            attn_hook = attn_module.register_forward_hook(make_attention_hook(layer_idx))
            self.hooks.append(attn_hook)

    def clear_outputs(self):
        """Clear output cache"""
        self.attention_outputs.clear()

    def remove_hooks(self):
        """Remove all hooks"""
        for hook in self.hooks:
            hook.remove()
        self.hooks.clear()

# ============== Forward calculation function ==============
@torch.no_grad()
def forward_with_hooks(inputs, model, hook_manager):
    """
    Execute forward pass and capture attention outputs and MLP residual-added outputs (hidden states)
    """
    hook_manager.clear_outputs()

    # Forward pass
    outputs = model(**inputs, output_hidden_states=True)

    # Get index of the last token
    last_idx = inputs.input_ids.shape[1] - 1

    # Determine number of layers
    language_model = model.model.language_model
    if hasattr(language_model, 'layers'):
        n_layers = len(language_model.layers)
    elif hasattr(language_model, 'decoder') and hasattr(language_model.decoder, 'layers'):
        n_layers = len(language_model.decoder.layers)
    else:
        n_layers = len(outputs.hidden_states) - 1

    # Extract attention output at last token position (after adding residual)
    attention_outputs = {}
    for layer_idx in range(n_layers):
        if layer_idx in hook_manager.attention_outputs:
            attn_output = hook_manager.attention_outputs[layer_idx][:, last_idx, :].squeeze(0)
            # hidden_states[layer_idx] is the input to the layer (the residual)
            residual = outputs.hidden_states[layer_idx][:, last_idx, :].squeeze(0).cpu()
            attn_plus_residual = attn_output + residual
            attention_outputs[layer_idx] = attn_plus_residual.float().cpu()

    # MLP output + residual is equivalent to the final hidden state of that layer
    mlp_outputs = {}
    for layer_idx in range(n_layers):
        # hidden_states[0] is embedding, hidden_states[1] is output of layer 0
        hidden_state = outputs.hidden_states[layer_idx + 1][:, last_idx, :].squeeze(0)
        mlp_outputs[layer_idx] = hidden_state.float().cpu()

    return attention_outputs, mlp_outputs, int(last_idx)

# ============== Data Utility Functions ==============
def iter_accepted_samples(path):
    """Read samples from accepted.jsonl"""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                yield obj
            except json.JSONDecodeError:
                continue

def build_messages(emotion: str, scenario: str, event: str):
    """Construct conversation messages"""
    system = f'''
Always reply in {emotion}.
Keep the reply to at most two sentences.
'''.strip()
    user = f"{scenario}\n{event}"
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

def prepare_inputs(tok, messages, device):
    """Apply chat template and move to device"""
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(device)
    return inputs

# ============== Save Function ==============
def save_group_outputs(group_data, attention_outputs, mlp_outputs, last_idx, attention_save_dir, mlp_save_dir):
    """Save all emotion outputs for a single group"""
    skeleton_id = group_data["skeleton_id"]
    valence = group_data["valence"]
    base = f"{skeleton_id}__{valence}"

    # Prepare Attention data
    attention_data = {
        "hidden_last_all_layers": {emotion: torch.stack([attention_outputs[emotion][i] for i in sorted(attention_outputs[emotion].keys())], dim=0).to(dtype=torch.float16, device="cpu") for emotion in EMOS6},
        "logits_0": {emotion: torch.zeros(1).to(dtype=torch.float16, device="cpu") for emotion in EMOS6},
        "input_ids": {emotion: torch.tensor([0], dtype=torch.int32) for emotion in EMOS6},
        "last_input_idx": {emotion: torch.tensor(last_idx, dtype=torch.int32) for emotion in EMOS6},
        "gen_text": {emotion: group_data["gen_texts"][emotion] for emotion in EMOS6}
    }

    # Prepare MLP data
    mlp_data = {
        "hidden_last_all_layers": {emotion: torch.stack([mlp_outputs[emotion][i] for i in sorted(mlp_outputs[emotion].keys())], dim=0).to(dtype=torch.float16, device="cpu") for emotion in EMOS6},
        "logits_0": {emotion: torch.zeros(1).to(dtype=torch.float16, device="cpu") for emotion in EMOS6},
        "input_ids": {emotion: torch.tensor([0], dtype=torch.int32) for emotion in EMOS6},
        "last_input_idx": {emotion: torch.tensor(last_idx, dtype=torch.int32) for emotion in EMOS6},
        "gen_text": {emotion: group_data["gen_texts"][emotion] for emotion in EMOS6}
    }

    # Save to disk
    attention_path = attention_save_dir / f"{base}.pt"
    torch.save(attention_data, attention_path)

    mlp_path = mlp_save_dir / f"{base}.pt"
    torch.save(mlp_data, mlp_path)

    return attention_path, mlp_path

# ============== Main Flow ==============
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="ministral3_3b", help="Model folder name")
    parser.add_argument("--model", type=str, default="mistralai/Ministral-3-3B-Instruct-2512", help="HuggingFace model name")
    parser.add_argument("--device", type=str, default="cuda:0", help="Device")
    parser.add_argument("--dtype", type=str, default="float16", choices=["float32","bfloat16","float16"], help="Data type")
    args = parser.parse_args()

    model_name = args.model_name
    dataset_name = "sev"

    # Define paths
    input_path = Path("/workspace/Various-Model/outputs") / model_name / "01_emotion_elicited_generation_prompt_based" / "labeled" / dataset_name / "accepted.jsonl"

    if not input_path.exists():
        print(f"[ERROR] Input file not found: {input_path}")
        return

    attention_save_dir = Path("/workspace/Various-Model/outputs") / model_name / "02_emotion_directions" / "residual_dump" / "attention"
    mlp_save_dir = Path("/workspace/Various-Model/outputs") / model_name / "02_emotion_directions" / "residual_dump" / "mlp"
    attention_save_dir.mkdir(parents=True, exist_ok=True)
    mlp_save_dir.mkdir(parents=True, exist_ok=True)

    # Dtype mapping
    dmap = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}
    torch_dtype = dmap[args.dtype]

    print("Loading model and tokenizer...")
    tok = AutoTokenizer.from_pretrained(args.model, use_fast=True, token=HF_TOKEN if HF_TOKEN else True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    model = Mistral3ForConditionalGeneration.from_pretrained(
        args.model,
        torch_dtype=torch_dtype,
        device_map=args.device,
        token=HF_TOKEN if HF_TOKEN else True
    )
    model.eval()

    print("Device:", model.device)
    print("Processing accepted samples...")

    hook_manager = AttentionHook(model)
    started = time.time()
    processed = 0
    skipped = 0

    # Group data container
    group_data = defaultdict(lambda: {
        "skeleton_id": None,
        "valence": None,
        "attention_outputs": {},
        "mlp_outputs": {},
        "gen_texts": {},
        "last_idx": None
    })

    try:
        for sample in iter_accepted_samples(input_path):
            key = sample["key"]
            skeleton_id = sample["skeleton_id"]
            valence = sample["valence"]
            emotion = sample["emotion"]
            group_key = f"{skeleton_id}__{valence}"

            # Skip already processed groups
            attention_path = attention_save_dir / f"{group_key}.pt"
            mlp_path = mlp_save_dir / f"{group_key}.pt"

            if attention_path.exists() and mlp_path.exists():
                skipped += 1
                if skipped % 50 == 0:
                    print(f"[SKIP] {skipped} groups skipped so far... (last: {group_key})")
                continue

            try:
                # Prepare inputs
                messages = build_messages(
                    sample["emotion"],
                    sample["scenario"],
                    sample["event"]
                )
                inputs = prepare_inputs(tok, messages, model.device)

                # Capture activations
                attn_out, mlp_out, last_idx = forward_with_hooks(inputs, model, hook_manager)

                # Collect into groups
                group_data[group_key]["skeleton_id"] = skeleton_id
                group_data[group_key]["valence"] = valence
                group_data[group_key]["attention_outputs"][emotion] = attn_out
                group_data[group_key]["mlp_outputs"][emotion] = mlp_out
                group_data[group_key]["gen_texts"][emotion] = sample["gen_text"]
                group_data[group_key]["last_idx"] = last_idx

                # Once all 6 emotions for a group are collected, save
                if len(group_data[group_key]["attention_outputs"]) == 6:
                    save_group_outputs(group_data[group_key],
                                     group_data[group_key]["attention_outputs"],
                                     group_data[group_key]["mlp_outputs"],
                                     group_data[group_key]["last_idx"],
                                     attention_save_dir,
                                     mlp_save_dir)

                    processed += 1
                    if processed % 10 == 0:
                        elapsed = time.time() - started
                        rate = processed / elapsed if elapsed > 0 else 0
                        print(f"[progress] processed={processed}, skipped={skipped}, elapsed={elapsed:.1f}s, rate={rate:.2f} groups/s")

                    # Clear memory for processed group
                    del group_data[group_key]

            except Exception as e:
                print(f"Error processing {key}: {e}")
                traceback.print_exc()
                continue

    finally:
        hook_manager.remove_hooks()

    elapsed = time.time() - started
    print(f"\n[OK] Done. processed={processed} groups, skipped={skipped}.")
    print(f"     Time: {elapsed:.1f}s | Rate: {processed/elapsed:.2f} groups/s")
    print(f"     Attention outputs saved to: {attention_save_dir}")
    print(f"     MLP outputs saved to: {mlp_save_dir}")

if __name__ == "__main__":
    main()

Loading model and tokenizer...


Loading weights:   0%|          | 0/822 [00:00<?, ?it/s]

Device: cuda:0
Processing accepted samples...
Found 26 layers in language_model.layers
[progress] processed=10, skipped=0, elapsed=3.9s, rate=2.57 groups/s
[progress] processed=20, skipped=0, elapsed=6.3s, rate=3.19 groups/s
[progress] processed=30, skipped=0, elapsed=8.5s, rate=3.53 groups/s
[progress] processed=40, skipped=0, elapsed=10.7s, rate=3.73 groups/s
[progress] processed=50, skipped=0, elapsed=13.0s, rate=3.86 groups/s
[progress] processed=60, skipped=0, elapsed=15.2s, rate=3.95 groups/s
[progress] processed=70, skipped=0, elapsed=17.4s, rate=4.02 groups/s
[progress] processed=80, skipped=0, elapsed=20.0s, rate=4.00 groups/s
[progress] processed=90, skipped=0, elapsed=22.4s, rate=4.01 groups/s
[progress] processed=100, skipped=0, elapsed=24.7s, rate=4.05 groups/s
[progress] processed=110, skipped=0, elapsed=26.9s, rate=4.09 groups/s
[progress] processed=120, skipped=0, elapsed=29.1s, rate=4.12 groups/s
[progress] processed=130, skipped=0, elapsed=31.7s, rate=4.10 groups/s
[p

In [2]:
import sys

sys.argv = [
    "program"
]


In [4]:
# scripts/02_emotion_direction_extraction/2_compute_emotion_directions.py
# -*- coding: utf-8 -*-
"""
Compute Emotion Direction Vectors - For Ministral-3-3B

Computes emotion direction vectors from residual_dump/attention/ and mlp/ respectively.

Method:
- Within each group: 6 emotions - group mean → de-contextualization
- Cross-group mean to get emotion centroids
- Remove global mean and normalize to get emotion direction vectors v_e^{(l)}
- Positions correspond to different intervention points in the residual stream

Input: outputs/{model_name}/02_emotion_directions/residual_dump/attention/ and mlp/
Output: outputs/{model_name}/02_emotion_directions/emo_directions_mlp.pt and emo_directions_attention.pt
"""

import os, argparse
from pathlib import Path
import numpy as np
import torch
import json
from collections import defaultdict

# 6 emotions
EMOS6 = ["anger", "sadness", "happiness", "fear", "surprise", "disgust"]

# Ministral-3-3B specific constants
MINISTRAL_3B_LAYERS = 26
MINISTRAL_3B_HIDDEN = 3072

def load_groups(data_dir: Path, data_type: str):
    """
    Load data of specified type, return groups: list[dict emo->[L,H]]

    Returns:
        groups: list of dicts, each dict has emotion -> numpy array [L, H]
        L: number of layers
        H: hidden dimension
    """
    pts = sorted(data_dir.glob("*.pt"))

    print(f"Loading {len(pts)} files for {data_type} emotion direction calculation...")

    # Group by skeleton_id and valence, collect all emotion data
    grouped_data = defaultdict(lambda: defaultdict(dict))

    loaded_count = 0
    for p in pts:
        try:
            rec = torch.load(p, map_location="cpu")
            loaded_count += 1
        except Exception as e:
            print(f"[LOAD-ERROR] {p.name}: {e}")
            continue

        H = rec.get("hidden_last_all_layers", {})
        if not H:
            print(f"[SKIP] {p.name}: no hidden_last_all_layers")
            continue

        # Extract skeleton_id and valence from filename (Format: skeleton_id__valence.pt)
        filename = p.stem
        if '__' not in filename:
            print(f"[SKIP] {p.name}: invalid filename format (no '__' separator)")
            continue

        skeleton_id, valence = filename.rsplit('__', 1)
        group_key = f"{skeleton_id}__{valence}"

        # Get emotion data from hidden_last_all_layers
        emotions_found = []
        for emotion, tensor in H.items():
            if emotion in EMOS6:
                if isinstance(tensor, torch.Tensor):
                    arr = tensor.numpy().astype(np.float32)
                    if len(arr.shape) == 2:
                        grouped_data[group_key][emotion] = arr
                        emotions_found.append(emotion)
                    else:
                        print(f"  Warning: {p.name} - {emotion} has unexpected shape {arr.shape}")

        if len(emotions_found) < 6:
            missing = [e for e in EMOS6 if e not in emotions_found]
            print(f"  Note: {p.name} has only {len(emotions_found)} emotions, missing {missing}")

    print(f"Successfully loaded {loaded_count} files")

    # Convert to groups format, keep only groups with all 6 emotions
    groups = []
    valid_groups = 0
    for group_key, emotions in grouped_data.items():
        if all(e in emotions for e in EMOS6):
            # Verify shapes are consistent
            first_emo = emotions[EMOS6[0]]
            consistent = True
            for e in EMOS6[1:]:
                if emotions[e].shape != first_emo.shape:
                    print(f"[ERROR] {group_key}: shape mismatch - {EMOS6[0]}: {first_emo.shape}, {e}: {emotions[e].shape}")
                    consistent = False
                    break

            if consistent:
                groups.append({e: emotions[e] for e in EMOS6})
                valid_groups += 1
        else:
            missing = [e for e in EMOS6 if e not in emotions]
            print(f"[SKIP] {group_key}: missing emotions {missing}")

    if not groups:
        raise RuntimeError(f"No usable groups found in {data_dir}")

    # Get dimensions from first group
    L = next(iter(groups))[EMOS6[0]].shape[0]
    Hdim = next(iter(groups))[EMOS6[0]].shape[1]

    # Validate against Ministral-3-3B specifications
    if L != MINISTRAL_3B_LAYERS:
        print(f"  Warning: Expected {MINISTRAL_3B_LAYERS} layers for Ministral-3-3B, but found {L}")
    if Hdim != MINISTRAL_3B_HIDDEN:
        print(f"  Warning: Expected hidden dim {MINISTRAL_3B_HIDDEN} for Ministral-3-3B, but found {Hdim}")

    print(f"Loaded {valid_groups} valid groups for {data_type} | layers={L}, hidden_dim={Hdim}")
    print(f"Rejected {len(grouped_data) - valid_groups} incomplete groups")

    return groups, L, Hdim

def build_emotion_directions(groups, L, Hdim):
    """
    Compute normalized emotion direction vectors
    """
    print("\nComputing emotion directions...")

    # Step 1: Remove within-group mean (decontextualization)
    print("  Step 1/4: Removing within-group means...")
    centered = []
    for g_idx, g in enumerate(groups):
        stacked = np.stack([g[e] for e in EMOS6], axis=0)
        mu = stacked.mean(axis=0)
        centered_g = {e: (g[e] - mu).astype(np.float32) for e in EMOS6}
        centered.append(centered_g)

        if (g_idx + 1) % 50 == 0:
            print(f"    Processed {g_idx + 1}/{len(groups)} groups")

    # Step 2: Cross-group emotion centroids
    print("  Step 2/4: Computing emotion centroids...")
    centroids = {}
    for e in EMOS6:
        e_stack = np.stack([cg[e] for cg in centered], axis=0)
        centroids[e] = e_stack.mean(axis=0)
        print(f"    {e:>9s} centroid shape: {centroids[e].shape}")

    # Step 3: Remove global mean
    print("  Step 3/4: Removing global mean...")
    all_centroids = np.stack(list(centroids.values()), axis=0)
    all_mean = all_centroids.mean(axis=0)

    # Step 4: Normalize to get direction vectors
    print("  Step 4/4: Normalizing to get direction vectors...")
    dirs = {}
    for e in EMOS6:
        v = centroids[e] - all_mean
        norms = np.linalg.norm(v, axis=1, keepdims=True)
        norms = np.maximum(norms, 1e-12)
        dirs[e] = (v / norms).astype(np.float32)

        avg_norm = np.mean(np.linalg.norm(dirs[e], axis=1))
        print(f"    {e:>9s} | shape={dirs[e].shape} | mean-norm={avg_norm:.4f}")

    return dirs

def compute_mlp_emotion_directions(mlp_dir, out_dir):
    """
    Compute MLP emotion direction vectors
    """
    print("\n" + "=" * 60)
    print("Computing MLP Emotion Directions")
    print("=" * 60)

    try:
        groups, L, Hdim = load_groups(mlp_dir, "MLP")
        dirs = build_emotion_directions(groups, L, Hdim)

        out_path = out_dir / "emo_directions_mlp.pt"
        metadata = {
            "model": "Ministral-3-3B",
            "num_groups": len(groups),
            "layers": L,
            "hidden_dim": Hdim,
            "emotions": EMOS6,
            "type": "mlp",
            "computation_date": str(np.datetime64('now')),
            "method": "group_mean_centering + global_mean_removal + normalization"
        }

        torch.save({"dirs": dirs, "metadata": metadata}, out_path)
        print(f"\n[✓] Saved MLP emotion directions to {out_path}")

        np_out_path = out_dir / "emo_directions_mlp.npz"
        np.savez(np_out_path, **{f"dirs_{e}": dirs[e] for e in EMOS6})
        print(f"[✓] Also saved as numpy array to {np_out_path}")
        return True

    except Exception as e:
        print(f"[ERROR] Failed to compute MLP emotion directions: {e}")
        import traceback
        traceback.print_exc()
        return False

def compute_attention_emotion_directions(attention_dir, out_dir):
    """
    Compute Attention emotion direction vectors
    """
    print("\n" + "=" * 60)
    print("Computing Attention Emotion Directions")
    print("=" * 60)

    try:
        groups, L, Hdim = load_groups(attention_dir, "Attention")
        dirs = build_emotion_directions(groups, L, Hdim)

        out_path = out_dir / "emo_directions_attention.pt"
        metadata = {
            "model": "Ministral-3-3B",
            "num_groups": len(groups),
            "layers": L,
            "hidden_dim": Hdim,
            "emotions": EMOS6,
            "type": "attention",
            "computation_date": str(np.datetime64('now')),
            "method": "group_mean_centering + global_mean_removal + normalization"
        }

        torch.save({"dirs": dirs, "metadata": metadata}, out_path)
        print(f"\n[✓] Saved Attention emotion directions to {out_path}")

        np_out_path = out_dir / "emo_directions_attention.npz"
        np.savez(np_out_path, **{f"dirs_{e}": dirs[e] for e in EMOS6})
        print(f"[✓] Also saved as numpy array to {np_out_path}")
        return True

    except Exception as e:
        print(f"[ERROR] Failed to compute Attention emotion directions: {e}")
        import traceback
        traceback.print_exc()
        return False

def analyze_directions(mlp_dirs, attention_dirs, out_dir):
    """
    Analyze and compare the computed directions
    """
    print("\n" + "=" * 60)
    print("Analysis of Emotion Directions")
    print("=" * 60)

    analysis = {"mlp": {}, "attention": {}, "comparison": {}}

    if mlp_dirs:
        print("\nMLP Directions Analysis:")
        mlp_cos_sim = {}
        for e1 in EMOS6:
            mlp_cos_sim[e1] = {}
            for e2 in EMOS6:
                if e1 != e2:
                    cos_sim = []
                    for l in range(mlp_dirs[e1].shape[0]):
                        v1, v2 = mlp_dirs[e1][l], mlp_dirs[e2][l]
                        cos = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-12)
                        cos_sim.append(cos)
                    mlp_cos_sim[e1][e2] = np.mean(cos_sim)
        analysis["mlp"]["cross_emotion_similarity"] = mlp_cos_sim
        for e1 in EMOS6:
            for e2 in EMOS6:
                if e1 < e2:
                    print(f"    {e1:>9s} - {e2:<9s}: {mlp_cos_sim[e1][e2]:.4f}")

    if attention_dirs:
        print("\nAttention Directions Analysis:")
        attn_cos_sim = {}
        for e1 in EMOS6:
            attn_cos_sim[e1] = {}
            for e2 in EMOS6:
                if e1 != e2:
                    cos_sim = []
                    for l in range(attention_dirs[e1].shape[0]):
                        v1, v2 = attention_dirs[e1][l], attention_dirs[e2][l]
                        cos = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-12)
                        cos_sim.append(cos)
                    attn_cos_sim[e1][e2] = np.mean(cos_sim)
        analysis["attention"]["cross_emotion_similarity"] = attn_cos_sim
        for e1 in EMOS6:
            for e2 in EMOS6:
                if e1 < e2:
                    print(f"    {e1:>9s} - {e2:<9s}: {attn_cos_sim[e1][e2]:.4f}")

    if mlp_dirs and attention_dirs:
        print("\nMLP vs Attention Comparison:")
        comparison = {}
        for e in EMOS6:
            cos_sim = []
            for l in range(mlp_dirs[e].shape[0]):
                v_mlp, v_attn = mlp_dirs[e][l], attention_dirs[e][l]
                cos = np.dot(v_mlp, v_attn) / (np.linalg.norm(v_mlp) * np.linalg.norm(v_attn) + 1e-12)
                cos_sim.append(cos)
            comparison[e] = np.mean(cos_sim)
        analysis["comparison"]["mlp_vs_attention"] = comparison
        for e in EMOS6:
            print(f"    {e:>9s}: {comparison[e]:.4f}")

    analysis_path = out_dir / "direction_analysis.json"
    with open(analysis_path, "w") as f:
        def convert_to_serializable(obj):
            if isinstance(obj, (np.float32, np.float64)): return float(obj)
            if isinstance(obj, dict): return {k: convert_to_serializable(v) for k, v in obj.items()}
            if isinstance(obj, list): return [convert_to_serializable(item) for item in obj]
            return obj
        json.dump(convert_to_serializable(analysis), f, indent=2)
    print(f"\n[✓] Saved analysis to {analysis_path}")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="ministral3_3b", help="Model folder name")
    parser.add_argument("--skip_analysis", action="store_true", help="Skip analysis")
    args = parser.parse_args()

    print("=" * 60)
    print("Computing Emotion Direction Vectors for Ministral-3-3B")
    print("=" * 60)

    model_name = args.model_name
    attention_dir = Path("/workspace/Various-Model/outputs") / model_name / "02_emotion_directions" / "residual_dump" / "attention"
    mlp_dir = Path("/workspace/Various-Model/outputs") / model_name / "02_emotion_directions" / "residual_dump" / "mlp"
    out_dir = Path("/workspace/Various-Model/outputs") / model_name / "02_emotion_directions"

    if not attention_dir.exists() or not mlp_dir.exists():
        print(f"[ERROR] Input directories not found. Run the dump script first.")
        return

    out_dir.mkdir(parents=True, exist_ok=True)

    mlp_success = compute_mlp_emotion_directions(mlp_dir, out_dir)
    attention_success = compute_attention_emotion_directions(attention_dir, out_dir)

    if not args.skip_analysis:
        mlp_dirs = torch.load(out_dir / "emo_directions_mlp.pt", weights_only=False)["dirs"] if mlp_success else None
        attn_dirs = torch.load(out_dir / "emo_directions_attention.pt", weights_only=False)["dirs"] if attention_success else None
        analyze_directions(mlp_dirs, attn_dirs, out_dir)

    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"MLP emotion directions: {'✓ SUCCESS' if mlp_success else '✗ FAILED'}")
    print(f"Attention emotion directions: {'✓ SUCCESS' if attention_success else '✗ FAILED'}")

if __name__ == "__main__":
    main()

Computing Emotion Direction Vectors for Ministral-3-3B

Computing MLP Emotion Directions
Loading 448 files for MLP emotion direction calculation...
Successfully loaded 448 files
Loaded 448 valid groups for MLP | layers=26, hidden_dim=3072
Rejected 0 incomplete groups

Computing emotion directions...
  Step 1/4: Removing within-group means...
    Processed 50/448 groups
    Processed 100/448 groups
    Processed 150/448 groups
    Processed 200/448 groups
    Processed 250/448 groups
    Processed 300/448 groups
    Processed 350/448 groups
    Processed 400/448 groups
  Step 2/4: Computing emotion centroids...
        anger centroid shape: (26, 3072)
      sadness centroid shape: (26, 3072)
    happiness centroid shape: (26, 3072)
         fear centroid shape: (26, 3072)
     surprise centroid shape: (26, 3072)
      disgust centroid shape: (26, 3072)
  Step 3/4: Removing global mean...
  Step 4/4: Normalizing to get direction vectors...
        anger | shape=(26, 3072) | mean-norm=1.0

In [1]:
import sys

sys.argv = [
    "program",
    "--dataset_name", "test_set",
    "--has_valence"
]


In [ ]:
# scripts/03_emotion_elicited_generation_steer_based/1_steer_with_emotion_direction.py
# -*- coding: utf-8 -*-
"""
Emotion-Steered Generation Script - Using Emotion Vectors to Guide Generation (Adapted for Ministral-3-3B)

Uses extracted emotion direction vectors to guide generation for input data.

Default parameters:
- layers: 11-20 (Layers to apply steering)
- alpha: 8 (Steering strength)
- last_k: 1 (Number of positions to steer)
- scale: rms (Scaling mode)
- dtype: float32 (Data type)
- direction_type: mlp (Direction type: mlp or attention)

Input: data/{dataset_name}.jsonl
Output: outputs/{model_name}/03_emotion_steered_generation/{dataset_name}/steered_outputs.jsonl
"""

import os, json, argparse, time
from pathlib import Path
import numpy as np
import torch
from transformers import AutoTokenizer, Mistral3ForConditionalGeneration
from huggingface_hub import login

# HF token (prioritize env var, otherwise use default login)
HF_TOKEN = os.environ.get('HF_TOKEN', None)
if HF_TOKEN:
    login(token=HF_TOKEN)

# 6 emotions and 3 valences
EMOS6 = ["anger","sadness","happiness","fear","surprise","disgust"]
VALENCES = ["positive", "neutral", "negative"]

# Ministral-3-3B specific constants
MINISTRAL_3B_LAYERS = 26
MINISTRAL_3B_HIDDEN = 3072

def build_messages(scenario: str, event: str):
    """
    Build conversation messages using standard chat template
    """
    system = "Keep the reply to at most two sentences."
    user = f"{scenario}\n{event}"
    return [{"role":"system","content":system},{"role":"user","content":user}]

def load_directions(path: Path):
    """
    Load emotion direction vectors from saved .pt file
    """
    obj = torch.load(path, map_location="cpu", weights_only=False)

    if "dirs" in obj:
        dirs = obj["dirs"]
        metadata = obj.get("metadata", {})
        L = metadata.get("layers", MINISTRAL_3B_LAYERS)
        H = metadata.get("hidden_dim", MINISTRAL_3B_HIDDEN)
        emos = metadata.get("emotions", EMOS6)
    else:
        dirs = obj["dirs"]
        L = obj["layers"]
        H = obj["hidden"]
        emos = obj["emotions"]

    for e in dirs:
        if not isinstance(dirs[e], np.ndarray):
            dirs[e] = np.array(dirs[e], dtype=np.float32)
        else:
            dirs[e] = dirs[e].astype(np.float32)

    return dirs, L, H, emos

def parse_layers(arg: str):
    """
    Parse layer range (e.g., '11-20') or comma-separated list
    """
    if "-" in arg:
        a, b = arg.split("-")
        return list(range(int(a), int(b) + 1))
    return [int(x) for x in arg.split(",") if x.strip()]

class EmotionSteerer:
    """
    Hook manager to apply steering vectors to specific layers during forward pass
    """
    def __init__(self, model, dirs_np, target_emotion, layer_ids, alpha=8.0, last_k=1, scale_mode="rms"):
        self.model = model
        self.alpha = float(alpha)
        self.layer_ids = list(layer_ids)
        self.last_k = int(last_k)
        self.scale_mode = scale_mode

        language_model = self.model.model.language_model

        if hasattr(language_model, 'layers'):
            self.layers = language_model.layers
        elif hasattr(language_model, 'decoder') and hasattr(language_model.decoder, 'layers'):
            self.layers = language_model.decoder.layers
        else:
            self.layers = None
            for name, module in language_model.named_children():
                if 'layer' in name.lower() and hasattr(module, '__len__'):
                    self.layers = module
                    break

        if self.layers is None:
            raise ValueError("Could not find transformer layers in the model")

        self.v = {}
        for l in self.layer_ids:
            if l < len(self.layers):
                self.v[l] = torch.from_numpy(dirs_np[target_emotion][l]).to(model.device)

        self.handles = []
        for l in self.layer_ids:
            if l in self.v:
                h = self.layers[l].register_forward_hook(self._make_hook(l))
                self.handles.append(h)

    def _make_hook(self, layer_id: int):
        v = self.v[layer_id]
        alpha = self.alpha
        last_k = self.last_k
        scale_mode = self.scale_mode

        def hook(module, inputs, output):
            if isinstance(output, (tuple, list)):
                if len(output) > 0:
                    hs = output[0].clone()
                    B, T, H = hs.shape
                    start = max(0, T - last_k)

                    if scale_mode == "rms":
                        seg = hs[:, start:T, :]
                        rms = torch.sqrt((seg**2).mean(dim=-1, keepdim=True) + 1e-12)
                        delta = alpha * v.view(1, 1, H) * rms
                    else:
                        delta = alpha * v.view(1, 1, H)

                    hs[:, start:T, :] = hs[:, start:T, :] + delta
                    return (hs,) + output[1:]
                return output
            else:
                hs = output.clone()
                B, T, H = hs.shape
                start = max(0, T - last_k)
                if scale_mode == "rms":
                    seg = hs[:, start:T, :]
                    rms = torch.sqrt((seg**2).mean(dim=-1, keepdim=True) + 1e-12)
                    delta = alpha * v.view(1, 1, H) * rms
                else:
                    delta = alpha * v.view(1, 1, H)
                hs[:, start:T, :] = hs[:, start:T, :] + delta
                return hs
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()
        self.handles.clear()

@torch.no_grad()
def generate_text(model, tok, messages, max_new_tokens=500):
    """
    Generate response using greedy decoding
    """
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    gen = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
        use_cache=True,
        min_new_tokens=10,
        repetition_penalty=1.05,
        no_repeat_ngram_size=3,
    )

    out_ids = gen[0][inputs.input_ids.shape[1]:]
    return tok.decode(out_ids, skip_special_tokens=True).strip()

def load_user_inputs(data_path: str, has_valence: bool = False):
    """
    Load JSONL input data
    """
    inputs = []
    with open(data_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                obj = json.loads(line)
                if has_valence:
                    if "event" in obj and "scenario" in obj:
                        event_data = obj["event"]
                        if all(v in event_data for v in VALENCES):
                            inputs.append(obj)
                else:
                    if "event" in obj and "scenario" in obj:
                        inputs.append(obj)
            except json.JSONDecodeError:
                continue
    return inputs
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_name", type=str, default="ministral3_3b")
    ap.add_argument("--model", type=str, default="mistralai/Ministral-3-3B-Instruct-2512")
    ap.add_argument("--dataset_name", type=str, default="test_set")
    ap.add_argument("--direction_type", type=str, default="mlp", choices=["mlp", "attention"])
    ap.add_argument("--layers", type=str, default="11-20")
    ap.add_argument("--alpha", type=float, default=8.0)
    ap.add_argument("--last_k", type=int, default=1)
    ap.add_argument("--scale", type=str, default="rms", choices=["unit","rms"])
    ap.add_argument("--dtype", type=str, default="float32", choices=["float16","bfloat16","float32"])
    ap.add_argument("--max_new_tokens", type=int, default=100)
    ap.add_argument("--data_path", type=str, default=None)
    ap.add_argument("--has_valence", action="store_true")
    ap.add_argument("--device", type=str, default="cuda:0")
    args = ap.parse_args()

    print("Emotion-Steered Generation Script (Ministral-3-3B)")
    print("=" * 60)

    # Paths
    model_name = args.model_name
    dataset_name = args.dataset_name
    directions_file = Path("/workspace/Various-Model/outputs") / model_name / "02_emotion_directions" / f"emo_directions_{args.direction_type}.pt"
    data_path = args.data_path if args.data_path else Path("/workspace/Various-Model/data") / f"{dataset_name}.jsonl"
    out_dir = Path("/workspace/Various-Model/outputs") / model_name / "03_emotion_steered_generation" / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)
    output_path = out_dir / "steered_outputs.jsonl"

    # Load directions
    if not directions_file.exists():
        print(f"[ERROR] Direction file not found: {directions_file}")
        return
    dirs, L, Hdim, emos = load_directions(directions_file)

    # Load dataset
    user_inputs = load_user_inputs(data_path, has_valence=args.has_valence)
    print(f"[data] Loaded {len(user_inputs)} inputs")

    # Load model
    torch_dtype = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}[args.dtype]
    tok = AutoTokenizer.from_pretrained(args.model, use_fast=True, token=HF_TOKEN if HF_TOKEN else True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    print("Loading Ministral-3-3B model...")
    model = Mistral3ForConditionalGeneration.from_pretrained(
        args.model,
        torch_dtype=torch_dtype,
        device_map=args.device,
        token=HF_TOKEN if HF_TOKEN else True
    )
    model.eval()

    # Steering
    layer_ids = parse_layers(args.layers)

    for i, user_input in enumerate(user_inputs):
        skeleton_id = user_input.get("skeleton_id", f"input_{i}")
        theme = user_input.get("theme", "Unknown")
        scenario = user_input["scenario"]

        print(f"[{i+1}/{len(user_inputs)}] {skeleton_id}")

        event_valences = {}

        # Choose valence keys
        if args.has_valence:
            valence_keys = VALENCES
        else:
            valence_keys = ["default"]

        for v_key in valence_keys:

            # Get event text
            if args.has_valence:
                if v_key not in user_input["event"]:
                    print(f"[WARN] Missing {v_key} in {skeleton_id}")
                    continue
                event_text = user_input["event"][v_key]
            else:
                event_text = user_input["event"]

            emotion_texts = {}

            for emo in EMOS6:
                steerer = EmotionSteerer(
                    model, dirs, emo, layer_ids,
                    alpha=args.alpha,
                    last_k=args.last_k,
                    scale_mode=args.scale
                )
                try:
                    msgs = build_messages(scenario, event_text)
                    emotion_texts[emo] = generate_text(model, tok, msgs, max_new_tokens=args.max_new_tokens)
                except Exception as e:
                    emotion_texts[emo] = f"[ERROR] {str(e)}"
                finally:
                    steerer.remove()

            event_valences[v_key] = emotion_texts

        # Save one JSON per sample
        result = {
            "skeleton_id": skeleton_id,
            "theme": theme,
            "scenario": scenario,
            "event_valences": event_valences,
            "parameters": {
                "layers": layer_ids,
                "alpha": args.alpha,
                "last_k": args.last_k,
                "scale": args.scale,
                "dtype": args.dtype,
                "direction_type": args.direction_type,
                "max_new_tokens": args.max_new_tokens,
                "sampling": "greedy"
            },
            "timestamp": int(time.time())
        }

        with open(output_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

    print(f"\n[OK] Saved to {output_path}")

if __name__ == "__main__":
    main()

Emotion-Steered Generation Script (Ministral-3-3B)
[data] Loaded 160 inputs
Loading Ministral-3-3B model...


Loading weights:   0%|          | 0/822 [00:00<?, ?it/s]

[1/160] work_test_00
[2/160] work_test_01
[3/160] work_test_02
[4/160] work_test_03
[5/160] work_test_04
[6/160] work_test_05
[7/160] work_test_06
[8/160] work_test_07
[9/160] work_test_08
[10/160] work_test_09
[11/160] work_test_10
[12/160] work_test_11
[13/160] work_test_12
[14/160] work_test_13
[15/160] work_test_14
[16/160] work_test_15
[35/160] school_test_14
[36/160] school_test_15


In [1]:
import sys

sys.argv = [
    "program",    
]

In [4]:
# scripts/03_emotion_elicited_generation_steer_based/2_label_steered_with_gpt.py
# -*- coding: utf-8 -*-
"""
GPT Labeling Script for Emotion-Steered Generated Text

Uses GPT-4o-mini to judge if generated text matches the target emotion.

Input: outputs/{model_name}/03_emotion_steered_generation/{dataset_name}/steered_outputs.jsonl
Output: outputs/{model_name}/03_emotion_steered_generation/{dataset_name}/labeled_results.jsonl
"""

import os, json, argparse, time, math, sys
from pathlib import Path
from typing import Dict, Any
from openai import OpenAI

# 6 emotions and 3 valences
EMOTIONS = ["anger","sadness","happiness","fear","surprise","disgust"]
VALENCES = ["positive", "neutral", "negative"]

# ========= OpenAI Client Setup =========
# Read API key from environment variable
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', None)
if not OPENAI_API_KEY:
    print("[WARNING] OPENAI_API_KEY not found. Please export it before running.")

client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

# GPT Prompts
SYSTEM = f'''
    You are a careful rater.
    Given a target emotion and a text,
    decide if the text's STYLE matches the target emotion among:
    {EMOTIONS}
    Focus on tone/attitude, not content valence.
'''

USER_TMPL = '''
    Target emotion: {emotion}
    Text:
    {text}
    Decide if the text's STYLE matches the target emotion.
    Return a compact JSON with keys exactly:
    {{
    "match": <0 or 1>,
    "reason": <short string>
    }}
'''

def extract_json_from_response(response: str) -> str:
    """
    Extract JSON content from GPT response, handling potential markdown blocks.
    """
    response = response.strip()
    if "```json" in response:
        start = response.find("```json") + 7
        end = response.find("```", start)
        if end != -1:
            return response[start:end].strip()
    elif "```" in response:
        start = response.find("```") + 3
        end = response.find("```", start)
        if end != -1:
            return response[start:end].strip()
    return response

def ask_llm(emotion: str, text: str, model: str, max_retries=4, backoff=1.8) -> Dict[str, Any]:
    """
    Call GPT to judge emotion matching with retry logic.
    """
    if client is None:
        return {"match": 0, "reason": "no-openai-client"}

    for i in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role":"system", "content": SYSTEM},
                    {"role":"user", "content": USER_TMPL.format(emotion=emotion, text=text)}
                ],
            )
            out = resp.choices[0].message.content
            print(f"      [{emotion}] GPT response captured.")

            json_str = extract_json_from_response(out)
            j = json.loads(json_str)

            # Ensure basic fields exist
            if "match" not in j: j = {"match": 0, "reason": "invalid-json"}
            if "reason" not in j: j["reason"] = "no-reason-provided"

            return j
        except Exception as e:
            print(f"      [{emotion}] Attempt {i+1} failed: {e}")
            if i == max_retries-1:
                return {"match": 0, "reason": f"error:{type(e).__name__}"}
            time.sleep(backoff ** i)

def get_processed_keys(output_file):
    """
    Get set of already processed sample keys to support resuming.
    """
    if not output_file.exists():
        return set()

    processed_keys = set()
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                obj = json.loads(line)
                key = f"{obj['skeleton_id']}___{obj.get('event_valence', 'neutral')}"
                processed_keys.add(key)
            except:
                continue
    return processed_keys
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="ministral3_3b")
    parser.add_argument("--dataset_name", type=str, default="test_set")
    parser.add_argument("--gpt_model", type=str, default="gpt-4o-mini")
    args = parser.parse_args()

    model_name = args.model_name
    dataset_name = args.dataset_name

    input_jsonl = Path("/workspace/Various-Model/outputs") / model_name / "03_emotion_steered_generation" / dataset_name / "steered_outputs.jsonl"
    output_dir = Path("/workspace/Various-Model/outputs") / model_name / "03_emotion_steered_generation" / dataset_name
    output_jsonl = output_dir / "labeled_results.jsonl"

    if not input_jsonl.exists():
        print(f"[ERROR] Missing input file: {input_jsonl}")
        sys.exit(1)

    if client is None:
        print("[ERROR] OpenAI client not initialized.")
        sys.exit(1)

    print("GPT Labeling for Emotion-Steered Generated Text")
    print("=" * 60)

    processed_keys = get_processed_keys(output_jsonl)

    with open(input_jsonl, "r", encoding="utf-8") as fin, \
         open(output_jsonl, "a", encoding="utf-8") as fout:

        for line_num, line in enumerate(fin, 1):
            line = line.strip()
            if not line:
                continue

            try:
                item = json.loads(line)
            except:
                continue

            skeleton_id = item["skeleton_id"]
            sample_key = skeleton_id
            if sample_key in processed_keys:
                continue

            print(f"\n[{line_num}] Processing {skeleton_id}")

            event_valences = item.get("event_valences", {})
            valence_labels = {}

            # Loop over positive/neutral/negative
            for valence in VALENCES:
                if valence not in event_valences:
                    continue

                print(f"  Valence: {valence}")
                emotion_texts = event_valences[valence]
                emotion_labels = {}

                # Loop over emotions
                for emotion in EMOTIONS:
                    text = emotion_texts.get(emotion, "")

                    if not isinstance(text, str) or text.strip() == "":
                        label = {"match": 0, "reason": "empty-text"}
                    else:
                        label = ask_llm(emotion, text, args.gpt_model)

                    emotion_labels[emotion] = label
                    time.sleep(0.1)

                valence_labels[valence] = emotion_labels

            # Build final output
            result = {
                "skeleton_id": skeleton_id,
                "theme": item.get("theme", ""),
                "scenario": item.get("scenario", ""),
                "parameters": item.get("parameters", {}),
                "valence_labels": valence_labels,
                "event_valences": event_valences,
                "event": item.get("event", ""),
                "timestamp": int(time.time())
            }

            fout.write(json.dumps(result, ensure_ascii=False) + "\n")
            fout.flush()

    print(f"\n[OK] Saved labeled file to {output_jsonl}")


if __name__ == "__main__":
    main()

GPT Labeling for Emotion-Steered Generated Text

[1] Processing work_test_00 - default
      [anger] GPT response captured.


KeyError: 'default'

In [7]:
import sys

sys.argv = [
    "program",
]

In [9]:
# scripts/03_emotion_elicited_generation_steer_based/3_generate_accuracy_stats.py
# -*- coding: utf-8 -*-
"""
Emotion-Steered Generation Accuracy Statistics Script

Reads GPT labeling results, calculates accuracy by emotion and valence

Input: outputs/{model_name}/03_emotion_steered_generation/{dataset_name}/labeled_results.jsonl
Output: outputs/{model_name}/03_emotion_steered_generation/{dataset_name}/accuracy_stats.json
"""

import json, argparse
from pathlib import Path
from collections import defaultdict

def generate_accuracy_stats(output_dir: Path, dataset_name: str):
    """
    Generate accuracy statistics for specified dataset
    """

    dataset_dir = output_dir / dataset_name
    labeled_path = dataset_dir / "labeled_results.jsonl"
    stats_path = dataset_dir / "accuracy_stats.json"

    if not labeled_path.exists():
        print(f"[ERROR] Labeled results file not found: {labeled_path}")
        return None

    # Statistics dictionaries
    stats_by_emotion = defaultdict(lambda: {"total": 0, "matches": 0})
    stats_by_valence = defaultdict(lambda: {"total": 0, "matches": 0})

    total_pairs = 0
    total_matches = 0

    # Read labeled_results
    print(f"Reading {labeled_path}...")
    with open(labeled_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)

                # valence_labels contains positive, neutral, negative keys
                valence_labels = item.get("valence_labels", {})

                # Iterate through each valence
                for event_valence, emotion_labels in valence_labels.items():
                    # Count matching results for each emotion
                    for emotion, label in emotion_labels.items():
                        match = label.get("match", 0)

                        stats_by_emotion[emotion]["total"] += 1
                        stats_by_valence[event_valence]["total"] += 1
                        total_pairs += 1

                        if match == 1:
                            stats_by_emotion[emotion]["matches"] += 1
                            stats_by_valence[event_valence]["matches"] += 1
                            total_matches += 1

            except json.JSONDecodeError:
                continue

    # Calculate accuracy
    for emotion in stats_by_emotion:
        total = stats_by_emotion[emotion]["total"]
        matches = stats_by_emotion[emotion]["matches"]
        stats_by_emotion[emotion]["accuracy"] = round(matches / total * 100, 2) if total > 0 else 0.0

    for valence in stats_by_valence:
        total = stats_by_valence[valence]["total"]
        matches = stats_by_valence[valence]["matches"]
        stats_by_valence[valence]["accuracy"] = round(matches / total * 100, 2) if total > 0 else 0.0

    # Overall statistics
    overall_accuracy = round(total_matches / total_pairs * 100, 2) if total_pairs > 0 else 0.0

    # Build statistics result
    stats = {
        "dataset": dataset_name,
        "overall": {
            "total_pairs": total_pairs,
            "matches": total_matches,
            "accuracy": overall_accuracy
        },
        "by_emotion": dict(stats_by_emotion),
        "by_valence": dict(stats_by_valence)
    }

    # Save statistics file
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    return stats, stats_path


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, default="ministral3_3b", help="Model folder name")
    parser.add_argument("--dataset", type=str, default="test_set", help="Dataset name (e.g., test_set, user_test)")
    parser.add_argument("--all", action="store_true", help="Process all datasets")
    args = parser.parse_args()

    # Determine output directory path
    output_dir = Path("/workspace/Various-Model/outputs") / args.model_name / "03_emotion_steered_generation"

    if not output_dir.exists():
        print(f"[ERROR] Output directory not found: {output_dir}")
        return

    # Determine datasets to process
    if args.all:
        # Auto-discover all dataset folders
        datasets = [d.name for d in output_dir.iterdir() if d.is_dir() and (d / "labeled_results.jsonl").exists()]
    elif args.dataset:
        datasets = [args.dataset]
    else:
        # Default: process all datasets with labeled_results.jsonl
        datasets = [d.name for d in output_dir.iterdir() if d.is_dir() and (d / "labeled_results.jsonl").exists()]

    if not datasets:
        print(f"[ERROR] No datasets with labeled_results.jsonl found in {output_dir}")
        return

    print("=" * 70)
    print("Generating Emotion Steering Accuracy Stats")
    print("=" * 70)

    for dataset_name in datasets:
        result = generate_accuracy_stats(output_dir, dataset_name)

        if result:
            stats, stats_path = result

            print(f"\n📊 {dataset_name.upper()} Dataset Statistics:")
            print(f"   File: {stats_path}")
            print(f"   Total Pairs: {stats['overall']['total_pairs']}")
            print(f"   Matches: {stats['overall']['matches']}")
            print(f"   Overall Accuracy: {stats['overall']['accuracy']}%")

            print(f"\n   By Emotion:")
            for emotion in sorted(stats['by_emotion'].keys()):
                e_stats = stats['by_emotion'][emotion]
                print(f"      {emotion:12s}: {e_stats['matches']:4d}/{e_stats['total']:4d} = {e_stats['accuracy']:6.2f}%")

            if stats['by_valence']:
                print(f"\n   By Valence:")
                for valence in sorted(stats['by_valence'].keys()):
                    v_stats = stats['by_valence'][valence]
                    print(f"      {valence:12s}: {v_stats['matches']:4d}/{v_stats['total']:4d} = {v_stats['accuracy']:6.2f}%")

    print("\n" + "=" * 70)
    print("✅ Statistics generation completed!")
    print("=" * 70)


if __name__ == "__main__":
    main()

Generating Emotion Steering Accuracy Stats
Reading /workspace/Various-Model/outputs/ministral3_3b/03_emotion_steered_generation/test_set/labeled_results.jsonl...

📊 TEST_SET Dataset Statistics:
   File: /workspace/Various-Model/outputs/ministral3_3b/03_emotion_steered_generation/test_set/accuracy_stats.json
   Total Pairs: 2880
   Matches: 2805
   Overall Accuracy: 97.4%

   By Emotion:
      anger       :  477/ 480 =  99.38%
      disgust     :  467/ 480 =  97.29%
      fear        :  442/ 480 =  92.08%
      happiness   :  480/ 480 = 100.00%
      sadness     :  460/ 480 =  95.83%
      surprise    :  479/ 480 =  99.79%

   By Valence:
      negative    :  945/ 960 =  98.44%
      neutral     :  931/ 960 =  96.98%
      positive    :  929/ 960 =  96.77%

✅ Statistics generation completed!
